## This notebook is for generate the structure to be used in the evaluation of the model trained
- For this evaluation we will use the model *ft:gpt-3.5-turbo-0125:ufcg::BTEw6z95* this model was trained with all of the 600 examples of the dataset

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from PIL import Image
import time
import os
import io
import base64
from openai import OpenAI
from dotenv import load_dotenv
import random
random.seed(42)
import json
import pandas as pd
import shutil
load_dotenv()

client = OpenAI(api_key = os.getenv('OPENAI_API_KEY'))

In [2]:
parameters = {
    "temperature": [0.5, 0.7],
    "learning goals": [
        {"en": "addition", "pt": "adição"},
        {"en": "subtraction", "pt": "subtração"},
        {"en": "multiplication", "pt": "multiplicação"},
        {"en": "division", "pt": "divisão"},
    ],
    "task": [
        "observe image, write answer",
        "observe image, draw answer",
        "observe images, write answer",
        "observe images, draw answer",
        "observe images, link objects",
    ],
    "themes": [
        {"en": "Circus", "pt": "Circo"},
        {"en": "Camp Forest", "pt": "Acampamento na Floresta"},
        {"en": "Safari", "pt": "Safari"},
        {"en": "Mail", "pt": "Correio"},
        {"en": "Robots and circuits", "pt": "Robôs e circuitos"},
        {"en": "Art studio", "pt": "Estúdio de Arte"},
    ],
}
MODEL = "ft:gpt-4.1-mini-2025-04-14:ufcg::C0UvUMGP"

In [3]:
PROMPT_PART_1 = \
f"""Create the complete structure of an **educational worksheet** designed to support student learning.

The worksheet must include a well-organized **visual layout**, using **HTML** to define structural elements and **CSS** to style and position content such as text, images, tables, and any other components that enhance the learning experience.

### 📌 Layout Requirements:

1. **Alignment with the Learning Task**  
   All elements must be positioned and formatted in a way that directly supports the question or task proposed.
   For example: if the student is asked to match images, the images should be arranged separately and spaced appropriately to allow for easy association.

2. **HTML Structure with Descriptive Attributes**  
   Each HTML element must include two key attributes:
   - `alt`: For all images, this attribute must contain a clear, descriptive text that explains the image in a way that allows the student to understand it even without visual access. The description should be contextually accurate and relevant to the task.
   - `exp`: A custom attribute added to each HTML tag that briefly explains the purpose of that element within the worksheet (e.g., `exp="Image illustrating photosynthesis for matching activity"`).

3. **CSS with Explanatory Comments**  
   All CSS used to style or position elements must include **comments above each class or rule**, explaining how the styling contributes to making the worksheet visually engaging, intuitive, and enjoyable for students.

4. **Image Sizing**  
   Images should be sized appropriately to fit within the printable area of the worksheet. They should not exceed the dimensions of the page and should be large enough to clearly represent the content they illustrate.
---

In addition to the HTML and CSS, you must generate a **JSON description** that outlines the worksheet’s structure and purpose. This JSON should contain the following fields:

- `task`: A clear explanation of the learning goal — what the student is expected to learn by completing the worksheet.
- `layout`: A detailed description of how the HTML elements are arranged and used, explaining where each component is placed and why.
- `answer`: A description of what the student must do to complete the worksheet successfully, including the correct answer and any necessary supporting information.

---

### ⚠️ Important Guidelines:

- **Do not generate interactive HTML.** The worksheet is meant to be printed and completed with pen and paper.
- **Ensure image sizes are appropriate** and do not exceed the printable area of the worksheet.
- Always begin by generating the **JSON description**, followed by the corresponding **HTML layout**.
- The elements like images, tables, and text must be arranged in a way that is **visually appealing** and **educationally effective**.
- The elements should be **aligned** and **spaced** to facilitate easy reading and interaction by the student.
- The worksheet should be **creative, engaging, and educational**, strictly adhering to the parameters provided.
- The worksheet should support **playful learning**, encouraging students to interact with the content visually and cognitively.

### 📌 Rules to Follow:

- Do **not** add any text beyond the JSON and the HTML structure of the worksheet.
- Make the worksheet **creative, engaging, and educational**, based strictly on the parameters provided.
- The worksheet should support **playful learning**, encouraging students to interact with the content visually and cognitively.


To create a new worksheet, use the following parameters:
"""

PROMPT_PART_2 = \
"""
Where:

- `level`: the intended school grade or year for the worksheet.
- `subject`: the academic subject to which the worksheet belongs.
- `theme`: the specific topic or concept being addressed.
- `sheet width`: the desired width of the printed worksheet (in pixels or units appropriate for styling).
- `task`: a concise instruction describing what the student must do to complete the worksheet.
- `language`: the language in which the worksheet should be written (e.g., "en" for English, "pt" for Portuguese). This parameter controls all the descriptions, explanations and instructions within the worksheet.
"""


In [4]:

def screenshot_completo_via_devtools(html_path, imagem_saida):
    options = Options()
    options.headless = True
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--hide-scrollbars")
    options.add_argument("--window-size=1920,1080")

    driver = webdriver.Chrome(options=options)
    
    file_url = "file://" + os.path.abspath(html_path)
    driver.get(file_url)

    # Espera carregamento (ajuste se necessário)
    driver.implicitly_wait(2)

    # Garante que está no topo
    driver.execute_script("window.scrollTo(0, 0)")

     # Mede o tamanho real do conteúdo (sem espaço vazio)
    largura = driver.execute_script("return document.documentElement.scrollWidth")
    altura = driver.execute_script("return document.documentElement.scrollHeight")

    # Redimensiona a janela para exatamente esse tamanho
    driver.set_window_size(1000, 1300)

    # Usa DevTools Protocol para capturar a página inteira
    screenshot = driver.execute_cdp_cmd("Page.captureScreenshot", {
        "format": "png",
        "captureBeyondViewport": True,
        "fromSurface": True
    })

    # Decodifica base64 e salva
    with open(imagem_saida, "wb") as f:
        f.write(base64.b64decode(screenshot['data']))

    driver.quit()
    print(f"Imagem salva em: {imagem_saida}")


# screenshot_completo_via_devtools("/home/matheus/Documentos/mestrado-matheus-lisboa/experimento2/runs/2025-07-06 14:08:09.507446.html", "saidase.png")

In [5]:
def make_request_gpt(parameters, model, temperature=1):

    prompt = f"""
{PROMPT_PART_1}

<parameters>

{PROMPT_PART_2}
"""

    para = ""
    for key, value in parameters.items():
        para += f"{key}: {value}\n"

    prompt = prompt.replace("<parameters>", para)

    completion = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {
                "role": "system",
                "content": "You are an assistant who helps a teacher create activities to help students learn",
            },
            {"role": "user", "content": prompt},
        ],
    )

    output = completion.choices[0].message.to_dict()["content"]

    return output


def extract_html_and_json(output):

    lines = []

    aux = False
    for line in output.split("\n"):
        if line == "<!DOCTYPE html>":
            aux = True
        if aux:
            lines.append(line)
        if line == "</html>":
            aux = False

    html_part = "\n".join(lines)
    return html_part


def composite_form(parameters):

    aux = """
1 - Discordo totalmente
2 - Discordo
3 - Neutro / Indiferente
4 - Concordo
5 - Concordo totalmente
"""
    aux2 = """
1 - Muito baixa – A atividade é altamente convencional, previsível e não apresenta nenhum elemento novo, original ou adaptado.
2 - Baixa – A atividade tem pequenas variações, mas ainda segue formatos comuns e pouco inovadores. Há esforço limitado para propor algo diferente.
3 - Moderada – A atividade apresenta algumas ideias criativas ou adaptações interessantes, mas ainda dentro de um formato tradicional.
4 - Alta – A atividade demonstra originalidade, com elementos inovadores e estratégias diferenciadas para envolver os alunos.
5 - Muito alta – A atividade é excepcionalmente criativa, integrando ideias originais, abordagens fora do comum e forte potencial de engajamento e impacto pedagógico.

"""

    q1 = f"""
A atividade trabalha adequadamente o conceito de {parameters["learning goals"]}?
{aux}
"""

    q2 = f"""
O enunciado da atividade é claro e compreensível?
{aux}
"""

    q3 = f"""
A atividade fornece as informações necessárias para que os alunos consigam
resolvê-la corretamente?
{aux}
"""
    q4 = f"""
A atividade está bem contextualizada dentro do tema {parameters["theme"]}?
{aux}
"""
    q5 = f"""
O layout da atividade é bem estruturado e facilita a compreensão?
{aux}
"""
    q6 = f"""
O nível da atividade é adequado para alunos do 2º ano do ensino fundamental?
{aux}
"""

    q7 = f"""
Você utilizaria essa atividade em sala de aula se ela incluísse imagens com as características descritas?
{aux}
"""

    q8 = f"""
A contextualização do tema é criativa e auxilia no aprendizado de forma lúdica?
{aux2}
"""

    q9 = f"""
A organização do leiaute da atividade é criativa e auxilia no aprendizado de forma lúdica?
{aux2}
"""

    q10 = f"""
A forma de responder à atividade (escrevendo, desenhando ou pintando) é criativa e auxilia no aprendizado de forma lúdica?
{aux2}
"""

    df = pd.DataFrame(
        [
            [q1, ""],
            [q2, ""],
            [q3, ""],
            [q4, ""],
            [q5, ""],
            [q6, ""],
            [q7, ""],
            [q8, ""],
            [q9, ""],
            [q10, ""],
        ],
        columns=["Pergunta", "Resposta"],
    )

    return df


def generate_dataset_evaluation(path_save, parameters):

    count = 1

    ## gerenate the worksheets from the model
    for temperature in parameters["temperature"]:

        for learning_goal in parameters["learning goals"]:
            for task in parameters["task"]:

                theme = random.choice(parameters["themes"])

                learning_goal_en = learning_goal["en"]
                learning_goal_pt = learning_goal["pt"]


                if learning_goal_en == "addition" or learning_goal_en == "multiplication":

                    path_save = f"{path_save}/person1"
                else:
                    path_save = f"{path_save}/person2"

                path_save = path_save.replace("//", "/")

                if not os.path.exists(path_save):
                    os.makedirs(path_save)

                shutil.copy("script_verify_answers.py", path_save)


                parameters_activity = {
                    "level": "2nd year of fundamental",
                    "learning goals": learning_goal_en,
                    "theme": theme,
                    "width-sheet": "800px",
                    "task": task,
                    "language": "pt",
                }

                print(
                    f"Generating activity {count} with parameters: {learning_goal_en}"
                )

                output = make_request_gpt(parameters_activity, MODEL, temperature)

                html_part = extract_html_and_json(output)

                # save the html to a file at tmp
                html_path = f"tmp/activity_{count}.html"
                os.makedirs(os.path.dirname(html_path), exist_ok=True)
                with open(html_path, "w") as f:
                    f.write(html_part)

                # take a screenshot of the html file
                imagem_saida = f"{path_save}/{learning_goal_pt}/{count}/example.png"
                os.makedirs(os.path.dirname(imagem_saida), exist_ok=True)
                screenshot_completo_via_devtools(html_path, imagem_saida)
                print(
                    f"Activity {count} generated and saved to {html_path} and {imagem_saida}"
                )

                # save the parameters
                parameters_activity_path = f"{path_save}/{learning_goal_pt}/{count}/parameters.json"
                os.makedirs(os.path.dirname(parameters_activity_path), exist_ok=True)
                parameters_activity["temperature"] = temperature
                parameters_activity["model"] = MODEL
                with open(parameters_activity_path, "w") as f:
                    json.dump(parameters_activity, f, indent=4)

                # save the form on the same folder
                form = composite_form(parameters_activity)

                form_path = f"{path_save}/{learning_goal_pt}/{count}/form.csv"
                form.to_csv(form_path, index=False)

                count += 1
    
    # select the worksheets from the dataset used to train

    

In [6]:
datenow = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
PATH_SAVE = f"/home/matheus/Documentos/mestrado-matheus-lisboa/experimento2/runs/dataset_evaluation/{datenow}/"
generate_dataset_evaluation(PATH_SAVE, parameters)

Generating activity 1 with parameters: addition
Imagem salva em: /home/matheus/Documentos/mestrado-matheus-lisboa/experimento2/runs/dataset_evaluation/2025-08-14 09:44:54/person1/adição/1/example.png
Activity 1 generated and saved to tmp/activity_1.html and /home/matheus/Documentos/mestrado-matheus-lisboa/experimento2/runs/dataset_evaluation/2025-08-14 09:44:54/person1/adição/1/example.png
Generating activity 2 with parameters: addition
Imagem salva em: /home/matheus/Documentos/mestrado-matheus-lisboa/experimento2/runs/dataset_evaluation/2025-08-14 09:44:54/person1/person1/adição/2/example.png
Activity 2 generated and saved to tmp/activity_2.html and /home/matheus/Documentos/mestrado-matheus-lisboa/experimento2/runs/dataset_evaluation/2025-08-14 09:44:54/person1/person1/adição/2/example.png
Generating activity 3 with parameters: addition
Imagem salva em: /home/matheus/Documentos/mestrado-matheus-lisboa/experimento2/runs/dataset_evaluation/2025-08-14 09:44:54/person1/person1/person1/adi

KeyboardInterrupt: 